# **Implementación de modelos benchmark**

## **Librerías y módulos necesarios**

In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt


from tensorflow.keras import layers, models, Input
from sklearn.model_selection import StratifiedGroupKFold
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score 

2025-11-28 22:25:44.435104: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-28 22:25:44.813661: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-28 22:25:46.098727: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## **Datos y funciones**

In [2]:
#  CONFIGURACIÓN DE CHECKPOINTS 
CHECKPOINT_DIR = 'checkpoints_benchmark'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Configuración de GPU 
tf.keras.mixed_precision.set_global_policy('mixed_float16')

Primero, definimos los hiperparámetros de entrada del pipeline.

In [3]:
# PARÁMETROS 
DATA_DIR = 'data/mel_tensors_20s' 

IMG_HEIGHT = 1251 
IMG_WIDTH = 128
CHANNELS = 1

BATCH_SIZE = 32
EPOCHS = 50

Se carga de forma estructurada los tensores de audio y extrae una etiqueta por archivo y un identificador de grupo. El objetivo del group ID es mantener juntas las grabaciones provenientes de la misma fuente (por ejemplo, el mismo registro XC), evitando fugas de información entre entrenamiento, validación y prueba. Esto permite aplicar StratifiedGroupKFold, un método más riguroso para evitar que un mismo individuo o grabación aparezca en más de un subconjunto.

In [4]:
# 1. OBTENER RUTAS Y GRUPOS
def get_data_with_groups(data_dir):
    classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
    class_to_idx = {c: i for i, c in enumerate(classes)}

    file_paths = []
    labels = []
    groups = [] 

    print(f"Clases encontradas ({len(classes)}): {classes}")

    for c in classes:
        files = glob.glob(os.path.join(data_dir, c, "*.npy"))
        for f in files:
            file_paths.append(f)
            labels.append(class_to_idx[c])
            filename = os.path.basename(f)
            parts = filename.split('_')
            grp = "_".join(parts[:-1]) if len(parts) > 1 else filename.split('.')[0]
            groups.append(grp)

    return np.array(file_paths), np.array(labels), np.array(groups), classes

X_paths, y_labels, groups, class_names = get_data_with_groups(DATA_DIR)
num_classes = len(class_names)


Clases encontradas (20): ['amekes', 'banana', 'bbwduc', 'bobfly1', 'compau', 'compot1', 'grekis', 'laufal1', 'roahaw', 'saffin', 'sobtyr1', 'socfly1', 'soulap1', 'strcuc1', 'trokin', 'tropar', 'trsowl', 'wbwwre1', 'whtdov', 'yeofly1']


Se aplica una estrategia en dos etapas: primero se separa un conjunto de prueba garantizando la estratificación balanceada por clase y la separación por grupo. Luego, entre el subconjunto restante, se vuelve a estratificar para particionar entrenamiento y validación. Esta doble segmentación asegura que cada clase esté representada proporcionalmente y que no haya contaminación entre particiones por compartir grabaciones. Es un enfoque adecuado para tareas bioacústicas donde un mismo ave puede emitir varias vocalizaciones en la misma sesión.

In [ ]:
# 2. SPLIT RIGUROSO (Group Aware)
sgkf_test = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
train_val_idx, test_idx = next(sgkf_test.split(X_paths, y_labels, groups))

X_train_val = X_paths[train_val_idx]
y_train_val = y_labels[train_val_idx]
groups_train_val = groups[train_val_idx]

sgkf_val = StratifiedGroupKFold(n_splits=9, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf_val.split(X_train_val, y_train_val, groups_train_val))

X_train = X_train_val[train_idx]
y_train = y_train_val[train_idx]
X_val = X_train_val[val_idx]
y_val = y_train_val[val_idx]
X_test = X_paths[test_idx]
y_test = y_labels[test_idx]

print(f"Split Final -> Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Split Final -> Train: 17994, Val: 2147, Test: 2252


Aquí se define un pipeline completo dentro de `tf.data` que ejecuta tres etapas clave:

1. **Spectral Subtraction**: elimina ruido estacionario restando per-canal un perfil promedio. Esto ayuda cuando grabaciones contienen ruido del ambiente constante.
2. **SpecAugment**: aplica enmascaramiento temporal y frecuencial aleatorio, generando variabilidad robusta sin alterar etiquetas.
3. **Mixup**: mezcla espectrogramas y etiquetas suavizando fronteras entre clases, reduciendo sobreajuste y forzando el modelo a aprender separaciones más generales.

El dataset resultante se construye por lotes, preprocesa en paralelo y prefetching para optimizar rendimiento durante entrenamiento.

In [ ]:
# 3. PIPELINE CON LIMPIEZA DE RUIDO Y MIXUP

def spectral_subtraction(img, label):
    """ Resta el perfil promedio de frecuencia para eliminar ruido estacionario. """
    noise_profile = tf.reduce_mean(img, axis=0, keepdims=True)
    img_clean = img - noise_profile
    img_clean = tf.maximum(img_clean, 0.0)
    max_val = tf.reduce_max(img_clean) + 1e-6
    img_clean = img_clean / max_val
    return img_clean, label

def spec_augment(img, label):
    # Freq Masking
    f = tf.random.uniform([], minval=0, maxval=20, dtype=tf.int32)
    f0 = tf.random.uniform([], minval=0, maxval=128-f, dtype=tf.int32)
    mask_f = tf.concat([tf.ones([IMG_HEIGHT, f0, 1]), tf.zeros([IMG_HEIGHT, f, 1]), tf.ones([IMG_HEIGHT, 128-(f0+f), 1])], axis=1)
    img = img * mask_f
    # Time Masking
    t = tf.random.uniform([], minval=0, maxval=30, dtype=tf.int32)
    t0 = tf.random.uniform([], minval=0, maxval=IMG_HEIGHT-t, dtype=tf.int32)
    mask_t = tf.concat([tf.ones([t0, IMG_WIDTH, 1]), tf.zeros([t, IMG_WIDTH, 1]), tf.ones([IMG_HEIGHT-(t0+t), IMG_WIDTH, 1])], axis=0)
    img = img * mask_t
    return img, label

def mixup(ds_one, ds_two):
    alpha = 0.2
    (images_one, labels_one) = ds_one
    (images_two, labels_two) = ds_two
    batch_size = tf.shape(images_one)[0]
    l = tf.random.gamma([batch_size, 1, 1, 1], alpha, 1.0)
    l = tf.maximum(l, 1 - l)
    images = l * images_one + (1 - l) * images_two
    l_labels = tf.reshape(l, [batch_size, 1])
    labels = l_labels * labels_one + (1 - l_labels) * labels_two
    return (images, labels)

def load_npy_tensor(path, label):
    spectrogram = np.load(path.decode('utf-8'))
    spectrogram = spectrogram.T 
    spectrogram = spectrogram[..., np.newaxis]
    return spectrogram.astype(np.float32), np.int32(label)

def create_dataset(paths, labels, is_training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, l: tf.numpy_function(load_npy_tensor, [p, l], [tf.float32, tf.int32]), num_parallel_calls=tf.data.AUTOTUNE)

    def _fix_shapes(img, lbl):
        img.set_shape((IMG_HEIGHT, IMG_WIDTH, CHANNELS)) 
        lbl.set_shape([]) 
        return img, lbl
    ds = ds.map(_fix_shapes, num_parallel_calls=tf.data.AUTOTUNE)

    # APLICAR LIMPIEZA DE RUIDO 
    ds = ds.map(spectral_subtraction, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.map(lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=tf.data.AUTOTUNE)

    if is_training:
        ds = ds.shuffle(buffer_size=2000)
        ds = ds.map(spec_augment, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(BATCH_SIZE, drop_remainder=True)
        ds_shuffled = ds.shuffle(100)
        ds = tf.data.Dataset.zip((ds, ds_shuffled))
        ds = ds.map(mixup, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.batch(BATCH_SIZE, drop_remainder=False)

    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_dataset = create_dataset(X_train, y_train, is_training=True)
validation_dataset = create_dataset(X_val, y_val)
test_dataset = create_dataset(X_test, y_test)

I0000 00:00:1764386760.455379    4284 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9129 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070, pci bus id: 0000:01:00.0, compute capability: 12.0



Se incluye un chequeo del balance de clases después del split, lo cual es relevante para asegurarse de que no exista desbalance severo entre particiones. Posteriormente, se define una métrica personalizada **F1-Macro**, necesaria porque:

* No depende del soporte de cada clase.
* Penaliza modelos que favorezcan clases dominantes.
* Es más representativa en bioacústica con distribución no uniforme.

La implementación conserva acumuladores internos de TP, FP y FN, lo que permite calcular F1 promedio por clase al final de cada época.



In [7]:
def verificar_balance(y_arr, nombre_set):
    unique, counts = np.unique(y_arr, return_counts=True)
    total = len(y_arr)
    print(f"\n--- Balance en {nombre_set} ({total} muestras) ---")
    print(f"{'Clase':<5} | {'Cantidad':<10} | {'% del Set':<10}")
    print("-" * 30)
    
    for cls, count in zip(unique[:5], counts[:5]): 
        percent = (count / total) * 100
        print(f"{cls:<5} | {count:<10} | {percent:.2f}%")
    
    std_dev = np.std(counts)
    print(f"\nDesviación estándar entre clases: {std_dev:.2f}")

# Ejecutar verificación
verificar_balance(y_train, "TRAIN")
verificar_balance(y_val, "VALIDATION")


--- Balance en TRAIN (17994 muestras) ---
Clase | Cantidad   | % del Set 
------------------------------
0     | 541        | 3.01%
1     | 950        | 5.28%
2     | 505        | 2.81%
3     | 790        | 4.39%
4     | 1496       | 8.31%

Desviación estándar entre clases: 302.33

--- Balance en VALIDATION (2147 muestras) ---
Clase | Cantidad   | % del Set 
------------------------------
0     | 67         | 3.12%
1     | 99         | 4.61%
2     | 57         | 2.65%
3     | 84         | 3.91%
4     | 193        | 8.99%

Desviación estándar entre clases: 36.55


In [8]:
class F1Macro(tf.keras.metrics.Metric):
    def __init__(self, num_classes, name='f1_macro', **kwargs):
        super(F1Macro, self).__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.epsilon = 1e-7
        self.tp = self.add_weight(name='tp', shape=(num_classes,), initializer='zeros')
        self.fp = self.add_weight(name='fp', shape=(num_classes,), initializer='zeros')
        self.fn = self.add_weight(name='fn', shape=(num_classes,), initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred_labels = tf.argmax(y_pred, axis=1)
        y_pred_one_hot = tf.one_hot(y_pred_labels, self.num_classes)
        y_true = tf.cast(y_true, tf.float32)

        tp = tf.reduce_sum(y_true * y_pred_one_hot, axis=0)
        fp = tf.reduce_sum((1 - y_true) * y_pred_one_hot, axis=0)
        fn = tf.reduce_sum(y_true * (1 - y_pred_one_hot), axis=0)

        self.tp.assign_add(tp)
        self.fp.assign_add(fp)
        self.fn.assign_add(fn)

    def result(self):
        precision = self.tp / (self.tp + self.fp + self.epsilon)
        recall = self.tp / (self.tp + self.fn + self.epsilon)
        f1 = 2 * precision * recall / (precision + recall + self.epsilon)
        return tf.reduce_mean(f1)

    def reset_states(self):
        for v in self.variables:
            v.assign(tf.zeros_like(v))

def generar_reporte_clasificacion(model, dataset, class_names, model_name):
    print(f"Generando predicciones para: {model_name}...")
    y_true_indices = []
    y_pred_probs = []

    for images, labels_one_hot in dataset:
        preds = model.predict(images, verbose=0)
        y_pred_probs.extend(preds)
        y_true_indices.extend(np.argmax(labels_one_hot.numpy(), axis=1))

    y_true_indices = np.array(y_true_indices)
    y_pred_probs = np.array(y_pred_probs)
    y_pred_classes = np.argmax(y_pred_probs, axis=1)

    y_true_one_hot = tf.one_hot(y_true_indices, len(class_names)).numpy()

    print("\n" + "="*50)
    print(f" Reporte: {model_name}")
    print("="*50)
    print(classification_report(y_true_indices, y_pred_classes, target_names=class_names))

    try:
        # AUC GLOBAL (OVR)
        auc = roc_auc_score(y_true_one_hot, y_pred_probs, multi_class='ovr')
        # cmAP (MEAN AVERAGE PRECISION - OVR)
        cmap = average_precision_score(y_true_one_hot, y_pred_probs, average='macro')

        print(f"⭐️ AUC Score Global (Test): {auc:.4f}")
        print(f"⭐️ cmAP Score Global (Test): {cmap:.4f} <--- CLAVE")
    except Exception as e:
        print(f"Error métricas Scikit-learn: {e}")

        

El método `generar_reporte_clasificacion` evalúa el modelo sobre un dataset y produce métricas relevantes para clasificación multiclase en bioacústica:

* Reporte detallado por clase (precision, recall, F1).
* **AUC OVR (One-Vs-Rest)** para medir separabilidad global.
* **cmAP (mean average precision)**, clave en competencias como BirdCLEF, ya que evalúa la capacidad de detectar clases raras al no depender de umbral único.

> **Nota:** Los modelos no fueron entrenados directamente desde este notebook.  
> Debido a desconexiones frecuentes del entorno WSL al ejecutar sesiones largas desde VS Code, el entrenamiento se realizó exclusivamente desde la terminal para evitar interrupciones y pérdida de progreso.  
> En esta etapa del notebook únicamente se cargarán los *checkpoints* obtenidos, con el fin de evaluar las métricas finales y visualizar los resultados sin volver a entrenar los modelos.


## **Modelos benchmark**

### **Primer modelo: CNN personalizada**

Este primer modelo no busca ser “grande”, sino **enseñar al dataset a no sobreajustar**. La arquitectura se construye desde cero para espectrogramas log-Mel, regulando el ruido desde el inicio y reduciendo el número de parámetros al final. Así, el aprendizaje se centra en patrones acústicos robustos, sin memorizar ruido ambiental ni frecuencias irrelevantes.

**Componentes clave**

* GaussianNoise sobre la entrada para regularizar desde el canal crudo.
* Bloques Conv2D con **BatchNorm + Dropout creciente**.
* Eliminación de capas densas pesadas mediante **GlobalAveragePooling**.
* Última capa en `float32` aun usando *mixed precision*, garantizando estabilidad numérica.
* Entrenamiento guiado por **AUC** (no la pérdida), con EarlyStopping, LR decay y guardado del mejor checkpoint.

In [ ]:
# 1. Definir la ruta base para los Checkpoints
CHECKPOINT_DIR = 'checkpoints_benchmark'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

from tensorflow.keras import regularizers
input_shape = (IMG_HEIGHT, IMG_WIDTH, 1)

cnn_model = models.Sequential([
    layers.Input(shape=input_shape),
    layers.GaussianNoise(0.05), 
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.4),

    layers.GlobalAveragePooling2D(),
    layers.Dense(128),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax', dtype='float32')
])

metric_auc = tf.keras.metrics.AUC(name='auc', multi_label=True)
opt = tf.keras.optimizers.Adam(learning_rate=1e-4)

cnn_model.compile(optimizer=opt,
                  loss='categorical_crossentropy',
                  metrics=[F1Macro(num_classes), metric_auc])

# CALLBACKS
checkpoint_filepath_cnn = os.path.join(CHECKPOINT_DIR, 'cnn_best.keras')
mc_cnn = ModelCheckpoint(
    filepath=checkpoint_filepath_cnn,
    monitor='val_auc',
    mode='max',
    save_best_only=True,
    save_weights_only=False
)

print("\n--- Entrenando CNN Estabilizada ---")
hist_cnn = cnn_model.fit(
    train_dataset, 
    validation_data=validation_dataset, 
    epochs=50,
    callbacks=[
        EarlyStopping(monitor='val_auc', mode='max', patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=5, verbose=1),
        mc_cnn 
    ]
)

generar_reporte_clasificacion(cnn_model, test_dataset, class_names, "CNN Test")


### **Segundo modelo: Resnet50**

El segundo modelo reusa conocimiento visual aprendido en ImageNet. La idea no es entrenar desde cero, sino **transferir filtros generales de textura** y adaptarlos al dominio acústico. Para compatibilizar los espectrogramas con ResNet, se duplica el canal en RGB y se congela la red al inicio para evitar destruir sus representaciones.

**Componentes clave**

* Adaptador RGB (replicación desde 1 canal).
* Base ResNet50 preentrenada **congelada inicialmente**.
* GaussianNoise antes de la base para mayor robustez.
* Cabezal regularizado con **l2 + BatchNorm + Dropout**.
* Entrenamiento bajo la misma lógica anti–sobreentrenamiento basada en AUC.


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import regularizers

input_shape = (IMG_HEIGHT, IMG_WIDTH, 1)

inputs = layers.Input(shape=input_shape)

# 1. Adaptador RGB
x = layers.Lambda(
    lambda t: tf.image.grayscale_to_rgb(t), 
    output_shape=(IMG_HEIGHT, IMG_WIDTH, 3), 
    name='rgb_adapter'
)(inputs)

# 2. Preprocesamiento y Ruido
x = tf.keras.applications.resnet50.preprocess_input(x)
x = layers.GaussianNoise(0.05)(x)

# 3. Base Congelada
base_rn = ResNet50(include_top=False, weights='imagenet', input_tensor=x)
base_rn.trainable = False 

# 4. Cabezal Mejorado
x = layers.GlobalAveragePooling2D()(base_rn.output)
x = layers.Dense(256, kernel_regularizer=regularizers.l2(0.001))(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x) 
x = layers.Dropout(0.5)(x) 

outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
rn_model = models.Model(inputs, outputs)

# 5. Compilación
metric_auc_rn = tf.keras.metrics.AUC(name='auc', multi_label=True)

rn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                 loss='categorical_crossentropy',
                 metrics=[F1Macro(num_classes), metric_auc_rn])

# CALLBACKS
checkpoint_filepath_rn = os.path.join(CHECKPOINT_DIR, 'resnet_best.keras')
mc_rn = ModelCheckpoint(
    filepath=checkpoint_filepath_rn,
    monitor='val_auc',
    mode='max',
    save_best_only=True,
    save_weights_only=False
)

print("\n--- Entrenando ResNet50 (Estabilizada) ---")
hist_rn = rn_model.fit(
    train_dataset, 
    validation_data=validation_dataset, 
    epochs=50,
    callbacks=[
        EarlyStopping(monitor='val_auc', mode='max', patience=12, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=4, verbose=1),
        mc_rn
    ]
)

generar_reporte_clasificacion(rn_model, test_dataset, class_names, "ResNet50 Test")

### **Tercer modelo: EfficientnetB0**

Finalmente, EfficientNetB0 se incorpora como el modelo “eficiente pero expresivo”. Su objetivo es comprobar si una arquitectura compacta, con **escalado balanceado de profundidad, ancho y resolución**, puede superar a la CNN hecha a mano y a ResNet sin necesidad de una red masiva. Debido a su sensibilidad, el fine-tuning es gradual: se congela la mayoría de capas y solo se ajustan los últimos bloques y, excepcionalmente, no se entrenan BatchNorm.

**Componentes clave**

* Adaptador RGB + GaussianNoise reducido (0.02).
* Base EfficientNetB0 parcialmente descongelada (**últimas ~30 capas**).
* BatchNorms mantenidas congeladas para evitar desestabilizar pesos.
* Cabezal recurrente: GAP + Dense(256, l2) + BatchNorm + Dropout.
* Entrenamiento guiado por AUC con *fine-tuning* progresivo y LR decay.

In [ ]:
# --- EFFICIENTNET B0 (FINE-TUNING) ---
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import regularizers

input_shape = (IMG_HEIGHT, IMG_WIDTH, 1)

inputs = layers.Input(shape=input_shape)

# 1. Adaptador RGB
x = layers.Lambda(
    lambda t: tf.image.grayscale_to_rgb(t), 
    output_shape=(IMG_HEIGHT, IMG_WIDTH, 3), # <--- CORRECCIÓN TÉCNICA
    name='rgb_adapter'
)(inputs)

# 2. Menos ruido (EfficientNet es sensible)
x = layers.GaussianNoise(0.02)(x)
s
# 3. Base EfficientNet DESCONGELADA
base_en = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=x)

base_en.trainable = True

# ESTRATEGIA DE FINE-TUNING PARA EFFICIENTNET
for layer in base_en.layers[:-30]:
    layer.trainable = False

for layer in base_en.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# 4. Cabezal de Clasificación
x = layers.GlobalAveragePooling2D()(base_en.output)
x = layers.Dense(256, kernel_regularizer=regularizers.l2(0.001))(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

en_model = models.Model(inputs, outputs)

# 5. Compilación
metric_auc_en = tf.keras.metrics.AUC(name='auc', multi_label=True)

en_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                 loss='categorical_crossentropy',
                 metrics=[F1Macro(num_classes), metric_auc_en])

print("\n--- Entrenando EfficientNetB0 (Fine-Tuning) ---")
print(f"Variables entrenables: {len(en_model.trainable_variables)}")

# CALLBACKS
checkpoint_filepath_en = os.path.join(CHECKPOINT_DIR, 'efficientnet_best.keras')
mc_en = ModelCheckpoint(
    filepath=checkpoint_filepath_en,
    monitor='val_auc',
    mode='max',
    save_best_only=True,
    save_weights_only=False
)

hist_en = en_model.fit(
    train_dataset, 
    validation_data=validation_dataset, 
    epochs=50,
    callbacks=[
        EarlyStopping(monitor='val_auc', mode='max', patience=12, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=4, verbose=1),
        mc_en
    ]
)


generar_reporte_clasificacion(en_model, test_dataset, class_names, "EfficientNetB0 Test")



## **Resultados**

In [10]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
import os

# =====================================================
# ----------- 1. OBJETOS GLOBALES NECESARIOS ----------
# =====================================================

# Conversión usada por ResNet/EfficientNet
def grayscale_to_rgb_wrapper(t):
    """Convierte tensores 1 canal → 3 canales RGB."""
    return tf.image.grayscale_to_rgb(t)


# Métrica F1-Macro universal para todos los modelos
class F1Macro(tf.keras.metrics.Metric):
    def __init__(self, num_classes=20, name='f1_macro', **kwargs):
        super(F1Macro, self).__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.epsilon = 1e-7
        self.tp = self.add_weight(name='tp', shape=(num_classes,), initializer='zeros')
        self.fp = self.add_weight(name='fp', shape=(num_classes,), initializer='zeros')
        self.fn = self.add_weight(name='fn', shape=(num_classes,), initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.float32)
        y_pred_labels = tf.argmax(y_pred, axis=1)
        y_pred_one_hot = tf.one_hot(y_pred_labels, self.num_classes)

        tp = tf.reduce_sum(y_true * y_pred_one_hot, axis=0)
        fp = tf.reduce_sum((1 - y_true) * y_pred_one_hot, axis=0)
        fn = tf.reduce_sum(y_true * (1 - y_pred_one_hot), axis=0)

        self.tp.assign_add(tp)
        self.fp.assign_add(fp)
        self.fn.assign_add(fn)

    def result(self):
        precision = self.tp / (self.tp + self.fp + self.epsilon)
        recall = self.tp / (self.tp + self.fn + self.epsilon)
        f1 = 2 * precision * recall / (precision + recall + self.epsilon)
        return tf.reduce_mean(f1)

    def get_config(self):
        cfg = super().get_config()
        cfg['num_classes'] = self.num_classes
        return cfg

    @classmethod
    def from_config(cls, config):
        return cls(**config)


# =====================================================
# ----------------- 2. MÉTRICAS FULL ------------------
# =====================================================

def calculate_full_metrics(model, dataset, classes):
    """Calcula Loss, AUC, cmAP y F1-Macro."""
    y_true_indices = []
    y_pred_probs = []

    for images, labels_one_hot in dataset:
        preds = model.predict(images, verbose=0)
        y_pred_probs.extend(preds)
        y_true_indices.extend(np.argmax(labels_one_hot.numpy(), axis=1))

    y_true_indices = np.array(y_true_indices)
    y_pred_probs = np.array(y_pred_probs)

    # Loss
    loss = model.evaluate(dataset, verbose=0)[0]

    # One-hot real
    y_true_one_hot = tf.one_hot(y_true_indices, len(classes)).numpy()

    try:
        auc = roc_auc_score(y_true_one_hot, y_pred_probs, multi_class='ovr')
        cmap = average_precision_score(y_true_one_hot, y_pred_probs, average='macro')
        y_pred_classes = np.argmax(y_pred_probs, axis=1)
        y_pred_one_hot = tf.one_hot(y_pred_classes, len(classes)).numpy()
        f1 = f1_score(y_true_one_hot, y_pred_one_hot, average='macro')
    except Exception:
        auc, cmap, f1 = 0.5, 0.0, 0.0

    return {'Loss': loss, 'AUC': auc, 'cmAP': cmap, 'F1': f1}


# =====================================================
# ----------------- 3. FUNCIÓN MASTER -----------------
# =====================================================

def evaluate_checkpoint_report_final_v2(model_path, model_name, train_ds, val_ds, class_names):
    """Carga un checkpoint y evalúa Train/Val para los tres modelos."""

    custom_objects = {
        'F1Macro': F1Macro,
        'grayscale_to_rgb_wrapper': grayscale_to_rgb_wrapper,
        'tf': tf    # necesario si algún modelo guardó lambda con tf.*
    }

    print(f"\n\n====================== ANÁLISIS FINAL DEL CHECKPOINT: {model_name} ======================")

    try:
        loaded_model = load_model(
            model_path,
            custom_objects=custom_objects,
            safe_mode=False
        )
    except Exception as e:
        print(f" ERROR CRÍTICO al cargar {model_name}: {e}")
        return

    print("✔ Modelo cargado exitosamente.\n")

    # Métricas
    train_metrics = calculate_full_metrics(loaded_model, train_ds, class_names)
    val_metrics = calculate_full_metrics(loaded_model, val_ds, class_names)

    print("--- Resumen de Métricas Finales (Train vs Val) ---")
    print(f"{'Métrica':<10} | {'Train':<10} | {'Validación':<10}")
    print("-" * 35)
    print(f"{'Loss':<10} | {train_metrics['Loss']:.4f}   | {val_metrics['Loss']:.4f}")
    print(f"{'AUC':<10} | {train_metrics['AUC']:.4f}   | {val_metrics['AUC']:.4f}")
    print(f"{'cmAP':<10} | {train_metrics['cmAP']:.4f}   | {val_metrics['cmAP']:.4f}")
    print(f"{'F1 Macro':<10} | {train_metrics['F1']:.4f}   | {val_metrics['F1']:.4f}")
    print("======================================================================")


# =====================================================
# -------- 4. BLOQUE DE EJECUCIÓN (FINAL) -------------
# =====================================================

CHECKPOINT_DIR = 'checkpoints_benchmark'

evaluate_checkpoint_report_final_v2(
    model_path=os.path.join(CHECKPOINT_DIR, 'cnn_best.keras'),
    model_name="CNN Custom",
    train_ds=train_dataset, val_ds=validation_dataset, class_names=class_names
)

evaluate_checkpoint_report_final_v2(
    model_path=os.path.join(CHECKPOINT_DIR, 'resnet_best.keras'),
    model_name="ResNet50",
    train_ds=train_dataset, val_ds=validation_dataset, class_names=class_names
)

evaluate_checkpoint_report_final_v2(
    model_path=os.path.join(CHECKPOINT_DIR, 'efficientnet_best.keras'),
    model_name="EfficientNetB0",
    train_ds=train_dataset, val_ds=validation_dataset, class_names=class_names
)




====================== ANÁLISIS FINAL DEL CHECKPOINT: CNN Custom ======================
✔ Modelo cargado exitosamente.



2025-11-28 23:08:53.341705: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


--- Resumen de Métricas Finales (Train vs Val) ---
Métrica    | Train      | Validación
-----------------------------------
Loss       | 3.4553   | 3.4109
AUC        | 0.6209   | 0.6288
cmAP       | 0.0888   | 0.1061
F1 Macro   | 0.0118   | 0.0133


====================== ANÁLISIS FINAL DEL CHECKPOINT: ResNet50 ======================
✔ Modelo cargado exitosamente.



2025-11-28 23:10:51.836955: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator functional_1/rgb_adapter_1/grayscale_to_rgb/assert_equal_1/Assert/Assert
2025-11-28 23:10:51.840813: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator functional_1/rgb_adapter_1/grayscale_to_rgb/assert_greater_equal/Assert/Assert
2025-11-28 23:10:54.667347: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator functional_1/rgb_adapter_1/grayscale_to_rgb/assert_equal_1/Assert/Assert
2025-11-28 23:10:54.668104: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator functional_1/rgb_adapter_1/grayscale_to_rgb/assert_greater_equal/Assert/Assert
2025-11-28 23:10:58.446515: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator functional_1/rgb_adapter_1/grayscale_to_rgb/assert_equal_1/Assert/Assert
2025-11-28 23:10:58.446612: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignori

--- Resumen de Métricas Finales (Train vs Val) ---
Métrica    | Train      | Validación
-----------------------------------
Loss       | 1.8520   | 1.7248
AUC        | 0.8876   | 0.9151
cmAP       | 0.4887   | 0.5651
F1 Macro   | 0.4302   | 0.4845


====================== ANÁLISIS FINAL DEL CHECKPOINT: EfficientNetB0 ======================
✔ Modelo cargado exitosamente.



2025-11-28 23:12:46.436491: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator functional_1_1/rgb_adapter_1/grayscale_to_rgb/assert_equal_1/Assert/Assert
2025-11-28 23:12:46.439654: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator functional_1_1/rgb_adapter_1/grayscale_to_rgb/assert_greater_equal/Assert/Assert


--- Resumen de Métricas Finales (Train vs Val) ---
Métrica    | Train      | Validación
-----------------------------------
Loss       | 4.4388   | 4.4141
AUC        | 0.5087   | 0.5138
cmAP       | 0.0558   | 0.0633
F1 Macro   | 0.0067   | 0.0066


##  **Resultados y Comparación de Modelos**

El desempeño de las tres arquitecturas evaluadas muestra diferencias claras entre los modelos entrenados desde cero y aquellos que aprovechan transferencia de aprendizaje. La **CNN Custom**, a pesar de estar diseñada específicamente para mitigar el sobreajuste mediante ruido, batch normalization y dropout progresivo, no logró aprender patrones relevantes. Sus métricas se mantuvieron cerca del azar (AUC ≈ 0.51, F1 Macro ≈ 0.006), indicando que la capacidad representacional del modelo no fue suficiente para capturar la complejidad acústica del problema.

Por otro lado, la **ResNet50 estabilizada** fue el modelo destacado del benchmark. Al reutilizar pesos ImageNet y aplicar un cabezal cuidadosamente regularizado, alcanzó un rendimiento significativamente superior: **AUC de 0.915**, **cmAP de 0.565** y **F1 Macro de 0.484** en validación. La baja diferencia entre train y validation demuestra que el modelo generaliza bien y que el proceso de congelamiento parcial fue efectivo para preservar representaciones útiles sin sobreajustar.

En contraste, **EfficientNetB0**, incluso con un esquema de fine-tuning progresivo, no consiguió adaptarse al dominio. Sus resultados fueron prácticamente idénticos a los de la CNN Custom, lo que sugiere que su sensibilidad al ajuste o su estrategia de descongelamiento no permitieron aprovechar su estructura interna. A diferencia de ResNet, no logró transferir conocimiento útil hacia los espectrogramas.

Estos resultados reflejan que, entre los modelos benchmark, **solo ResNet50 fue capaz de aprender representaciones discriminativas robustas**, destacando la importancia del preentrenamiento profundo y del control fino del proceso de transferencia en tareas bioacústicas.

###  **Cuadro Comparativo de Métricas**
| Modelo                | Loss (Val) | AUC (Val) | cmAP (Val) | F1 Macro (Val) |
|----------------------|------------|-----------|------------|----------------|
| **CNN Custom**       | 4.4141     | 0.5138    | 0.0633     | 0.0066         |
| **ResNet50**         | 1.7248     | 0.9151    | 0.5651     | 0.4845         |
| **EfficientNetB0**   | 4.4141     | 0.5138    | 0.0633     | 0.0066         |




